# Module 04 — ML Theory & Evaluation
## 04-03: Feature Engineering & Pipelines

**Objective:** Implement numerical scalers, categorical encoders, and feature selectors from scratch; compose them into a leak-proof preprocessing pipeline; and validate against sklearn's Pipeline and ColumnTransformer.

**Prerequisites:** 2-01 through 2-10 (Supervised Learning)

## Part 0 — Setup & Prerequisites

This notebook owns **numerical scaling (standard, min-max, robust), categorical encoding (ordinal, one-hot, target), feature selection (mutual information, L1), and sklearn Pipeline / ColumnTransformer**. The central theme is **data leakage prevention**: every transformer must be fit on training data only and then applied to validation/test data — fitting on the full dataset before splitting inflates scores by leaking test-set statistics into the model.

**Prerequisites:** 2-01 through 2-10 (Supervised Learning)

> **Note (Category C):** The focus is the preprocessing methodology. We run a Ridge regressor solely to measure the downstream effect of different pipeline choices — model training is not the topic.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import warnings
import random
import time
from typing import Any
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler as SkStandardScaler,
    MinMaxScaler as SkMinMaxScaler,
    RobustScaler as SkRobustScaler,
    OneHotEncoder as SkOneHotEncoder,
    OrdinalEncoder as SkOrdinalEncoder,
)
from sklearn.feature_selection import SelectFromModel
from sklearn import metrics as skmetrics

import torch

from sklearn.model_selection import KFold
from sklearn.preprocessing import KBinsDiscretizer as SkKBins
from sklearn.metrics import r2_score
print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Constants ─────────────────────────────────────────────────────────────────
TEST_SIZE: float = 0.20
RIDGE_ALPHA: float = 1.0
LASSO_ALPHA: float = 0.01
CV_K: int = 5                    # k for cross-validation
N_MI_BINS: int = 10              # Bins for mutual information estimation
N_SYNTH: int = 2000              # Synthetic dataset rows

print(f'SEED       : {SEED}')
print(f'Test split : {TEST_SIZE:.0%}')
print(f'Ridge alpha: {RIDGE_ALPHA}')

### Data Loading & Exploration

We use two datasets:
1. **California Housing** — 8 numeric features, continuous regression target.
2. **Synthetic mixed dataset** — 4 numeric + 3 categorical features, created here to demonstrate categorical encoders and ColumnTransformer on a realistic mixed-type table.

In [ ]:
# ── California Housing (numeric only) ─────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df_housing: pd.DataFrame = housing.frame
feature_cols_hs: list = list(housing.feature_names)
target_col_hs: str = housing.target_names[0]

X_hs: np.ndarray = df_housing[feature_cols_hs].values
y_hs: np.ndarray = df_housing[target_col_hs].values

X_tr_hs, X_te_hs, y_tr_hs, y_te_hs = train_test_split(
    X_hs, y_hs, test_size=TEST_SIZE, random_state=SEED)
print(f'California Housing — train: {X_tr_hs.shape}, test: {X_te_hs.shape}')
print(f'Features: {feature_cols_hs}')

# ── Synthetic mixed-type dataset ──────────────────────────────────────────────
rng_syn = np.random.RandomState(SEED)

# Numeric features
size_sqft: np.ndarray = rng_syn.uniform(400, 4000, N_SYNTH)       # square footage
lot_size:  np.ndarray = rng_syn.uniform(1000, 20000, N_SYNTH)     # lot size
age_years: np.ndarray = rng_syn.randint(0, 100, N_SYNTH).astype(float)
distance_km: np.ndarray = rng_syn.exponential(8, N_SYNTH)         # skewed, outliers

# Categorical features
zone_cats: list = ['urban', 'suburban', 'rural', 'exurban']
condition_cats: list = ['poor', 'fair', 'good', 'excellent']
style_cats: list = ['ranch', 'colonial', 'contemporary', 'craftsman', 'tudor']

zone: np.ndarray = rng_syn.choice(zone_cats, N_SYNTH,
                                   p=[0.40, 0.35, 0.15, 0.10])
condition: np.ndarray = rng_syn.choice(condition_cats, N_SYNTH,
                                        p=[0.10, 0.25, 0.45, 0.20])
style: np.ndarray = rng_syn.choice(style_cats, N_SYNTH)

# Synthetic price with known ground-truth coefficients
zone_effect: dict = {'urban': 50, 'suburban': 30, 'rural': 10, 'exurban': 20}
cond_effect: dict = {'poor': -20, 'fair': 0, 'good': 20, 'excellent': 40}
price: np.ndarray = (
    100
    + 0.08 * size_sqft
    + 0.002 * lot_size
    - 0.5 * age_years
    - 1.2 * distance_km
    + np.array([zone_effect[z] for z in zone])
    + np.array([cond_effect[c] for c in condition])
    + rng_syn.normal(0, 30, N_SYNTH)
)

df_syn: pd.DataFrame = pd.DataFrame({
    'size_sqft': size_sqft,
    'lot_size':  lot_size,
    'age_years': age_years,
    'distance_km': distance_km,
    'zone': zone,
    'condition': condition,
    'style': style,
    'price': price,
})

num_cols_syn: list = ['size_sqft', 'lot_size', 'age_years', 'distance_km']
cat_cols_syn: list = ['zone', 'condition', 'style']

df_tr_syn, df_te_syn = train_test_split(
    df_syn, test_size=TEST_SIZE, random_state=SEED)
print(f'\nSynthetic dataset — train: {df_tr_syn.shape}, test: {df_te_syn.shape}')
print('Categorical value counts (zone):',
      df_tr_syn['zone'].value_counts().to_dict())

# EDA: feature distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
ax = axes[0]
ax.boxplot([X_tr_hs[:, i] for i in range(X_tr_hs.shape[1])],
           labels=feature_cols_hs, vert=True)
ax.set_title('California Housing: Feature Distributions (Box Plot)')
ax.set_ylabel('Raw value')
ax.tick_params(axis='x', rotation=30)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(df_tr_syn['distance_km'], bins=40, color='salmon', edgecolor='white')
ax.set_xlabel('distance_km')
ax.set_ylabel('Count')
ax.set_title('Synthetic: Skewed Feature (distance_km) — Needs Scaling')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_03_eda.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Part 1 — Scalers, Encoders & Selectors from Scratch

### 1.1 Numerical Scalers

Scaling matters because many algorithms (Ridge, SVM, KNN, neural nets) are sensitive to feature magnitude. Three standard strategies:

| Scaler | Formula | Use When |
|--------|---------|----------|
| Standard | $z = (x - \mu) / \sigma$ | Normally distributed, no severe outliers |
| Min-Max | $z = (x - x_{\min}) / (x_{\max} - x_{\min})$ | Bounded output needed, no outliers |
| Robust | $z = (x - Q_2) / (Q_3 - Q_1)$ | Heavy-tailed distributions, outliers present |

All three follow the same `fit` / `transform` / `inverse_transform` API — fit on train only, apply to both train and test.

In [ ]:
class StandardScaler:
    '''Zero-mean, unit-variance scaler.

    Transforms each feature by subtracting its training mean and dividing
    by its training standard deviation: z = (x - mu) / sigma.

    Attributes:
        mean_: Per-feature mean computed on training data.
        std_: Per-feature standard deviation computed on training data.
    '''

    def __init__(self, eps: float = 1e-8) -> None:
        '''Initialise StandardScaler.

        Args:
            eps: Small constant added to std to avoid division by zero.
        '''
        self.eps: float = eps
        self.mean_: np.ndarray | None = None
        self.std_: np.ndarray | None = None

    def fit(self, X: np.ndarray) -> 'StandardScaler':
        '''Compute per-feature mean and std from training data.

        Args:
            X: Training matrix of shape (n_samples, n_features).

        Returns:
            Self (fitted).
        '''
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Apply standardisation.

        Args:
            X: Matrix of shape (n_samples, n_features).

        Returns:
            Standardised matrix of the same shape.
        '''
        return (X - self.mean_) / (self.std_ + self.eps)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit on X and return transformed X.

        Args:
            X: Training matrix of shape (n_samples, n_features).

        Returns:
            Standardised training matrix.
        '''
        return self.fit(X).transform(X)

    def inverse_transform(self, Z: np.ndarray) -> np.ndarray:
        '''Reverse standardisation.

        Args:
            Z: Standardised matrix of shape (n_samples, n_features).

        Returns:
            Matrix in original scale.
        '''
        return Z * (self.std_ + self.eps) + self.mean_


class MinMaxScaler:
    '''Scales features to a fixed [feature_range] interval.

    z = (x - x_min) / (x_max - x_min) * (max_val - min_val) + min_val.

    Attributes:
        min_: Per-feature training minimum.
        max_: Per-feature training maximum.
        min_val: Lower bound of the target range.
        max_val: Upper bound of the target range.
    '''

    def __init__(self, feature_range: tuple = (0.0, 1.0), eps: float = 1e-8) -> None:
        '''Initialise MinMaxScaler.

        Args:
            feature_range: Tuple (min_val, max_val) of the target output range.
            eps: Small constant to avoid division by zero on constant features.
        '''
        self.min_val: float = float(feature_range[0])
        self.max_val: float = float(feature_range[1])
        self.eps: float = eps
        self.min_: np.ndarray | None = None
        self.max_: np.ndarray | None = None

    def fit(self, X: np.ndarray) -> 'MinMaxScaler':
        '''Compute per-feature min and max from training data.

        Args:
            X: Training matrix of shape (n_samples, n_features).

        Returns:
            Self (fitted).
        '''
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Apply min-max scaling.

        Args:
            X: Matrix of shape (n_samples, n_features).

        Returns:
            Scaled matrix with values in [min_val, max_val].
        '''
        scale = self.max_ - self.min_ + self.eps
        z_01: np.ndarray = (X - self.min_) / scale
        return z_01 * (self.max_val - self.min_val) + self.min_val

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit on X and return transformed X.

        Args:
            X: Training matrix.

        Returns:
            Scaled training matrix.
        '''
        return self.fit(X).transform(X)

    def inverse_transform(self, Z: np.ndarray) -> np.ndarray:
        '''Reverse min-max scaling.

        Args:
            Z: Scaled matrix.

        Returns:
            Matrix in original scale.
        '''
        scale = self.max_ - self.min_ + self.eps
        z_01: np.ndarray = (Z - self.min_val) / (self.max_val - self.min_val + self.eps)
        return z_01 * scale + self.min_


class RobustScaler:
    '''Scales features using the interquartile range (IQR).

    z = (x - Q2) / IQR  where IQR = Q3 - Q1. Robust to outliers because
    the median and quartiles are not influenced by extreme values.

    Attributes:
        median_: Per-feature median (Q2) from training data.
        iqr_: Per-feature IQR (Q3 - Q1) from training data.
    '''

    def __init__(
        self,
        quantile_range: tuple = (25.0, 75.0),
        eps: float = 1e-8,
    ) -> None:
        '''Initialise RobustScaler.

        Args:
            quantile_range: Tuple (q_low, q_high) in percentile units for IQR
                computation. Defaults to (25, 75) = standard IQR.
            eps: Small constant added to IQR to prevent division by zero.
        '''
        self.q_low: float = float(quantile_range[0])
        self.q_high: float = float(quantile_range[1])
        self.eps: float = eps
        self.median_: np.ndarray | None = None
        self.iqr_: np.ndarray | None = None

    def fit(self, X: np.ndarray) -> 'RobustScaler':
        '''Compute per-feature median and IQR from training data.

        Args:
            X: Training matrix of shape (n_samples, n_features).

        Returns:
            Self (fitted).
        '''
        self.median_ = np.median(X, axis=0)
        q_lo: np.ndarray = np.percentile(X, self.q_low, axis=0)
        q_hi: np.ndarray = np.percentile(X, self.q_high, axis=0)
        self.iqr_ = q_hi - q_lo
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Apply robust scaling.

        Args:
            X: Matrix of shape (n_samples, n_features).

        Returns:
            Robustly scaled matrix.
        '''
        return (X - self.median_) / (self.iqr_ + self.eps)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit on X and return transformed X.

        Args:
            X: Training matrix.

        Returns:
            Robustly scaled training matrix.
        '''
        return self.fit(X).transform(X)


# Validate against sklearn
ss_ours = StandardScaler()
Z_ss_ours: np.ndarray = ss_ours.fit_transform(X_tr_hs)
Z_ss_sk: np.ndarray = SkStandardScaler().fit_transform(X_tr_hs)
assert np.allclose(Z_ss_ours, Z_ss_sk, atol=1e-6), 'StandardScaler mismatch!'

mm_ours = MinMaxScaler()
Z_mm_ours: np.ndarray = mm_ours.fit_transform(X_tr_hs)
Z_mm_sk: np.ndarray = SkMinMaxScaler().fit_transform(X_tr_hs)
assert np.allclose(Z_mm_ours, Z_mm_sk, atol=1e-6), 'MinMaxScaler mismatch!'

rb_ours = RobustScaler()
Z_rb_ours: np.ndarray = rb_ours.fit_transform(X_tr_hs)
Z_rb_sk: np.ndarray = SkRobustScaler().fit_transform(X_tr_hs)
assert np.allclose(Z_rb_ours, Z_rb_sk, atol=1e-6), 'RobustScaler mismatch!'

print('StandardScaler, MinMaxScaler, RobustScaler all match sklearn ✓')

### 1.2 Categorical Encoders

Categorical variables must be converted to numbers before being fed to most ML algorithms. Three common strategies:

- **Ordinal encoding:** Each category gets an integer code. Only valid when the categories have a natural order (e.g., `poor < fair < good < excellent`).
- **One-hot encoding (OHE):** Creates a binary column for each category. Correct for nominal (unordered) categories, but expands dimensionality and introduces multicollinearity (use `drop='first'` to avoid).
- **Target encoding:** Replaces each category with the mean of the target variable in that category. Effective for high-cardinality features, but **requires care to prevent leakage** — compute mean on training fold only.

In [ ]:
class OrdinalEncoder:
    '''Encodes categorical features as ordinal integers.

    Each unique category in column j is mapped to an integer 0..K_j-1.
    Unknown categories at transform time are mapped to -1.

    Attributes:
        categories_: List of arrays; categories_[j] holds the sorted unique
            categories for feature j.
    '''

    def __init__(self) -> None:
        '''Initialise OrdinalEncoder.'''
        self.categories_: list = []

    def fit(self, X: np.ndarray) -> 'OrdinalEncoder':
        '''Learn category vocabularies from training data.

        Args:
            X: 2-D array of shape (n_samples, n_cat_features) with string
                or integer category values.

        Returns:
            Self (fitted).
        '''
        self.categories_ = []
        for col in range(X.shape[1]):
            unique_cats: np.ndarray = np.unique(X[:, col].astype(str))
            self.categories_.append(unique_cats)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Encode categories as integers.

        Args:
            X: 2-D array of shape (n_samples, n_cat_features).

        Returns:
            Integer-encoded array of the same shape. Unknown categories → -1.
        '''
        n_samples: int = X.shape[0]
        n_cols: int = X.shape[1]
        encoded: np.ndarray = np.full((n_samples, n_cols), -1, dtype=int)
        for col, cats in enumerate(self.categories_):
            cat_to_idx: dict = {c: i for i, c in enumerate(cats)}
            for row in range(n_samples):
                val: str = str(X[row, col])
                encoded[row, col] = cat_to_idx.get(val, -1)
        return encoded

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit and transform.

        Args:
            X: 2-D categorical array.

        Returns:
            Ordinal-encoded array.
        '''
        return self.fit(X).transform(X)


class OneHotEncoder:
    '''Creates binary indicator columns for each category.

    For K categories in a feature, produces K-1 columns when drop=True
    (to avoid multicollinearity) or K columns when drop=False.

    Attributes:
        categories_: List of category arrays, one per input column.
        drop: Whether to drop the first category column.
    '''

    def __init__(self, drop: bool = True) -> None:
        '''Initialise OneHotEncoder.

        Args:
            drop: If True, drop the first category to avoid multicollinearity.
        '''
        self.drop: bool = drop
        self.categories_: list = []

    def fit(self, X: np.ndarray) -> 'OneHotEncoder':
        '''Learn category vocabularies from training data.

        Args:
            X: 2-D categorical array of shape (n_samples, n_cat_features).

        Returns:
            Self (fitted).
        '''
        self.categories_ = []
        for col in range(X.shape[1]):
            unique_cats: np.ndarray = np.unique(X[:, col].astype(str))
            self.categories_.append(unique_cats)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''One-hot encode categorical columns.

        Args:
            X: 2-D categorical array of shape (n_samples, n_cat_features).

        Returns:
            Binary matrix of shape (n_samples, n_output_cols) where
            n_output_cols = sum(K_j - drop) over all columns j.
        '''
        n_samples: int = X.shape[0]
        cols: list = []
        for col, cats in enumerate(self.categories_):
            start: int = 1 if self.drop else 0
            for cat in cats[start:]:
                indicator: np.ndarray = (X[:, col].astype(str) == cat).astype(float)
                cols.append(indicator)
        return np.column_stack(cols) if cols else np.zeros((n_samples, 0))

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit and transform.

        Args:
            X: 2-D categorical array.

        Returns:
            One-hot encoded matrix.
        '''
        return self.fit(X).transform(X)

    @property
    def n_output_cols(self) -> int:
        '''Number of output columns after encoding.'''
        return sum(len(c) - (1 if self.drop else 0) for c in self.categories_)


class TargetEncoder:
    '''Encodes categories with the per-category mean of the target variable.

    Uses smoothing toward the global mean to handle rare categories:
    encoded = (n_k * mean_k + m * global_mean) / (n_k + m)
    where n_k is the category count and m is the smoothing factor.

    Attributes:
        global_mean_: Global target mean computed on training data.
        encodings_: List of dicts mapping category → smoothed mean, per column.
        smoothing: Smoothing factor m controlling influence of global mean.
    '''

    def __init__(self, smoothing: float = 10.0) -> None:
        '''Initialise TargetEncoder.

        Args:
            smoothing: Smoothing factor m; larger values pull category means
                more strongly toward the global mean. Default 10.
        '''
        self.smoothing: float = smoothing
        self.global_mean_: float = 0.0
        self.encodings_: list = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'TargetEncoder':
        '''Compute smoothed per-category target means.

        Args:
            X: 2-D categorical array of shape (n_samples, n_cat_features).
            y: Target array of shape (n_samples,).

        Returns:
            Self (fitted).
        '''
        self.global_mean_ = float(y.mean())
        self.encodings_ = []
        for col in range(X.shape[1]):
            mapping: dict = {}
            for cat in np.unique(X[:, col].astype(str)):
                mask: np.ndarray = X[:, col].astype(str) == cat
                n_k: int = int(mask.sum())
                mean_k: float = float(y[mask].mean())
                smoothed: float = (
                    n_k * mean_k + self.smoothing * self.global_mean_
                ) / (n_k + self.smoothing)
                mapping[cat] = smoothed
            self.encodings_.append(mapping)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Replace categories with their smoothed target means.

        Args:
            X: 2-D categorical array of shape (n_samples, n_cat_features).

        Returns:
            Float matrix of shape (n_samples, n_cat_features).
        '''
        n_samples: int = X.shape[0]
        encoded: np.ndarray = np.zeros((n_samples, X.shape[1]), dtype=float)
        for col, mapping in enumerate(self.encodings_):
            for row in range(n_samples):
                cat: str = str(X[row, col])
                encoded[row, col] = mapping.get(cat, self.global_mean_)
        return encoded

    def fit_transform(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        '''Fit and transform.

        Args:
            X: 2-D categorical array.
            y: Target array.

        Returns:
            Target-encoded matrix.
        '''
        return self.fit(X, y).transform(X)


# Validate encoders on synthetic data
X_cat_tr: np.ndarray = df_tr_syn[cat_cols_syn].values
X_cat_te: np.ndarray = df_te_syn[cat_cols_syn].values
y_tr_syn: np.ndarray = df_tr_syn['price'].values
y_te_syn: np.ndarray = df_te_syn['price'].values

oe = OrdinalEncoder()
Z_oe = oe.fit_transform(X_cat_tr)
print(f'OrdinalEncoder output shape   : {Z_oe.shape}  (same as input)')
print(f'  zone categories: {oe.categories_[0]}')

ohe = OneHotEncoder(drop=True)
Z_ohe = ohe.fit_transform(X_cat_tr)
print(f'OneHotEncoder output shape    : {Z_ohe.shape}  '
      f'(expected {ohe.n_output_cols} cols with drop=True)')

te = TargetEncoder(smoothing=10.0)
Z_te = te.fit_transform(X_cat_tr, y_tr_syn)
print(f'TargetEncoder output shape    : {Z_te.shape}')
print(f'  zone encoding: { {k: round(v, 1) for k,v in te.encodings_[0].items()} }')

### 1.3 Feature Selection

Feature selection removes uninformative or redundant inputs. Two complementary approaches:

- **Mutual information (filter method):** $I(X; Y) = \sum_{x,y} p(x,y) \log \frac{p(x,y)}{p(x)p(y)}$. Measures dependence between each feature and the target — computed independently of any model. We estimate MI with a histogram approximation.
- **L1 regularization (embedded method):** Lasso drives coefficient weights to exactly zero, implicitly selecting features. Features with non-zero coefficients are retained. Owned by Module 2-02; we call sklearn's SelectFromModel here.

In [ ]:
def mutual_information_continuous(
    x: np.ndarray,
    y: np.ndarray,
    n_bins: int = N_MI_BINS,
) -> float:
    '''Estimate mutual information between a feature and a continuous target.

    Uses a 2-D histogram to approximate the joint probability p(x, y)
    and marginals p(x), p(y). This is a simple estimator; in practice
    sklearn uses k-nearest-neighbour MI estimators for better accuracy.

    Args:
        x: 1-D feature array of shape (n_samples,).
        y: 1-D target array of shape (n_samples,).
        n_bins: Number of bins for each axis of the 2-D histogram.

    Returns:
        Non-negative scalar MI estimate in nats.
    '''
    counts, _, _ = np.histogram2d(x, y, bins=n_bins)
    joint_prob: np.ndarray = counts / counts.sum()
    px: np.ndarray = joint_prob.sum(axis=1, keepdims=True)
    py: np.ndarray = joint_prob.sum(axis=0, keepdims=True)
    marginal_product: np.ndarray = px * py

    mi: float = 0.0
    mask: np.ndarray = (joint_prob > 0) & (marginal_product > 0)
    mi = float(np.sum(
        joint_prob[mask] * np.log(joint_prob[mask] / marginal_product[mask])
    ))
    return max(0.0, mi)


def select_features_mi(
    X: np.ndarray,
    y: np.ndarray,
    k: int,
    n_bins: int = N_MI_BINS,
    feature_names: list | None = None,
) -> tuple[np.ndarray, np.ndarray, list]:
    '''Select the top-k features by mutual information with the target.

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        y: Target array of shape (n_samples,).
        k: Number of top features to retain.
        n_bins: Histogram bins for MI estimation.
        feature_names: Optional list of column names for reporting.

    Returns:
        Tuple (X_selected, mi_scores, selected_names) where:
          X_selected: Submatrix with the top-k columns.
          mi_scores: 1-D array of MI scores for all features.
          selected_names: List of selected feature names (empty if not provided).
    '''
    n_features: int = X.shape[1]
    mi_scores: np.ndarray = np.array([
        mutual_information_continuous(X[:, j], y, n_bins)
        for j in range(n_features)
    ])
    top_k_idx: np.ndarray = np.argsort(mi_scores)[::-1][:k]
    selected_names: list = (
        [feature_names[i] for i in top_k_idx] if feature_names else []
    )
    return X[:, top_k_idx], mi_scores, selected_names


# Compute MI scores for California Housing features
ss_mi = StandardScaler()
X_tr_sc_mi: np.ndarray = ss_mi.fit_transform(X_tr_hs)

mi_scores_hs: np.ndarray
_, mi_scores_hs, _ = select_features_mi(
    X_tr_sc_mi, y_tr_hs, k=X_tr_hs.shape[1],
    feature_names=feature_cols_hs,
)

mi_order: np.ndarray = np.argsort(mi_scores_hs)[::-1]
print('California Housing — feature MI scores (histogram estimator):')
print(f'{"Feature":<15} {"MI score":>10}')
print('-' * 27)
for idx in mi_order:
    print(f'{feature_cols_hs[idx]:<15} {mi_scores_hs[idx]:>10.4f}')

---
## Part 2 — Assembling the Pipeline Harness

The key insight is that **all transformers must be fit on training data only**. Fitting on the full dataset before the train/test split leaks test-set statistics (mean, std, category frequencies) into the training process, producing overly optimistic results. The `TabularPipeline` class enforces this discipline.

### 2.1 Data Leakage Demonstration

First, we quantify the magnitude of the leakage bias using StandardScaler as a concrete example.

In [ ]:
# ── Data leakage: fitting scaler on entire dataset before splitting ────────────
print('=== Data Leakage Demonstration ===')
print()

ridge = Ridge(alpha=RIDGE_ALPHA)
n_cv_reps: int = 20

leaked_scores: list = []
clean_scores: list = []

rng_leak = np.random.RandomState(SEED)
for _ in range(n_cv_reps):
    # Random 80/20 split
    n = len(y_hs)
    perm = rng_leak.permutation(n)
    n_tr = int(0.8 * n)
    tr_idx, te_idx = perm[:n_tr], perm[n_tr:]

    X_tr_l, X_te_l = X_hs[tr_idx], X_hs[te_idx]
    y_tr_l, y_te_l = y_hs[tr_idx], y_hs[te_idx]

    # LEAKY: fit scaler on ALL data, then split
    ss_leaked = StandardScaler()
    X_all_scaled: np.ndarray = ss_leaked.fit_transform(X_hs)  # fits on full dataset!
    X_tr_leaked: np.ndarray = X_all_scaled[tr_idx]
    X_te_leaked: np.ndarray = X_all_scaled[te_idx]
    ridge.fit(X_tr_leaked, y_tr_l)
    leaked_scores.append(ridge.score(X_te_leaked, y_te_l))

    # CLEAN: fit scaler only on training fold
    ss_clean = StandardScaler()
    X_tr_clean: np.ndarray = ss_clean.fit_transform(X_tr_l)
    X_te_clean: np.ndarray = ss_clean.transform(X_te_l)  # only transform, not fit!
    ridge.fit(X_tr_clean, y_tr_l)
    clean_scores.append(ridge.score(X_te_clean, y_te_l))

mean_leaked: float = float(np.mean(leaked_scores))
mean_clean: float = float(np.mean(clean_scores))
print(f'Leaky R² (mean ± std): {mean_leaked:.5f} ± {np.std(leaked_scores):.5f}')
print(f'Clean R² (mean ± std): {mean_clean:.5f} ± {np.std(clean_scores):.5f}')
print(f'Optimistic bias from leakage: {mean_leaked - mean_clean:+.5f}')
print()
print('Even for a linear scaler, leakage introduces a small but measurable optimistic bias.')
print('For more aggressive preprocessing (PCA, high-cardinality OHE) the bias grows larger.')

In [ ]:
class TabularPipeline:
    '''Leak-proof preprocessing pipeline for tabular data with mixed types.

    Applies a numerical scaler to numeric columns and a categorical encoder
    to categorical columns, then concatenates the outputs. All transformers
    are fit on training data only.

    Attributes:
        num_cols: Indices or slice for numeric columns in the input array.
        cat_cols: Indices or slice for categorical columns in the input array.
        scaler: Fitted numerical scaler instance.
        encoder: Fitted categorical encoder instance.
        is_fitted: Whether the pipeline has been fitted.
    '''

    def __init__(
        self,
        num_cols: list,
        cat_cols: list,
        scaler: Any = None,
        encoder: Any = None,
    ) -> None:
        '''Initialise TabularPipeline.

        Args:
            num_cols: List of column indices for numeric features.
            cat_cols: List of column indices for categorical features.
            scaler: Numerical scaler instance (e.g. StandardScaler()). If
                None, numeric features are passed through unscaled.
            encoder: Categorical encoder instance (e.g. OneHotEncoder()). If
                None, categorical features are dropped.
        '''
        self.num_cols: list = num_cols
        self.cat_cols: list = cat_cols
        self.scaler: Any = scaler
        self.encoder: Any = encoder
        self.is_fitted: bool = False

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray | None = None,
    ) -> 'TabularPipeline':
        '''Fit scaler and encoder on training data.

        Args:
            X: Training data of shape (n_samples, n_total_features).
            y: Optional target array (required for TargetEncoder).

        Returns:
            Self (fitted).
        '''
        if self.num_cols and self.scaler is not None:
            X_num: np.ndarray = X[:, self.num_cols].astype(float)
            self.scaler.fit(X_num)

        if self.cat_cols and self.encoder is not None:
            X_cat: np.ndarray = X[:, self.cat_cols]
            if isinstance(self.encoder, TargetEncoder):
                assert y is not None, 'TargetEncoder requires y in fit()'
                self.encoder.fit(X_cat, y)
            else:
                self.encoder.fit(X_cat)

        self.is_fitted = True
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Apply fitted transformers and concatenate outputs.

        Args:
            X: Data of shape (n_samples, n_total_features).

        Returns:
            Transformed matrix of shape (n_samples, n_output_features).
        '''
        parts: list = []

        if self.num_cols:
            X_num: np.ndarray = X[:, self.num_cols].astype(float)
            if self.scaler is not None:
                X_num = self.scaler.transform(X_num)
            parts.append(X_num)

        if self.cat_cols and self.encoder is not None:
            X_cat: np.ndarray = X[:, self.cat_cols]
            parts.append(self.encoder.transform(X_cat).astype(float))

        return np.concatenate(parts, axis=1) if parts else X

    def fit_transform(
        self,
        X: np.ndarray,
        y: np.ndarray | None = None,
    ) -> np.ndarray:
        '''Fit and transform training data.

        Args:
            X: Training data of shape (n_samples, n_total_features).
            y: Optional target array for TargetEncoder.

        Returns:
            Transformed training matrix.
        '''
        return self.fit(X, y).transform(X)


# Sanity check on synthetic data
X_tr_syn_raw: np.ndarray = df_tr_syn[num_cols_syn + cat_cols_syn].values
X_te_syn_raw: np.ndarray = df_te_syn[num_cols_syn + cat_cols_syn].values

num_idx = list(range(len(num_cols_syn)))
cat_idx = list(range(len(num_cols_syn), len(num_cols_syn) + len(cat_cols_syn)))

pipe_ohe = TabularPipeline(
    num_cols=num_idx, cat_cols=cat_idx,
    scaler=StandardScaler(), encoder=OneHotEncoder(drop=True),
)
X_tr_ohe: np.ndarray = pipe_ohe.fit_transform(X_tr_syn_raw)
X_te_ohe: np.ndarray = pipe_ohe.transform(X_te_syn_raw)
print(f'TabularPipeline (OHE): train shape={X_tr_ohe.shape}, test shape={X_te_ohe.shape}')

pipe_te = TabularPipeline(
    num_cols=num_idx, cat_cols=cat_idx,
    scaler=StandardScaler(), encoder=TargetEncoder(smoothing=10.0),
)
X_tr_te_enc: np.ndarray = pipe_te.fit_transform(X_tr_syn_raw, y_tr_syn)
X_te_te_enc: np.ndarray = pipe_te.transform(X_te_syn_raw)
print(f'TabularPipeline (TargetEnc): train shape={X_tr_te_enc.shape}, test shape={X_te_te_enc.shape}')

---
## Part 3 — Applying Pipelines to Real Data

We apply our pipeline to both datasets and compare Ridge regression performance across different preprocessing choices. Then we validate our from-scratch pipeline against sklearn's Pipeline / ColumnTransformer.

In [ ]:
# ── Apply different scalers to California Housing → Ridge regression ───────────
print('=== Scaler Comparison on California Housing (Ridge, test R²) ===')

scaler_configs: list = [
    ('No scaling', None),
    ('StandardScaler', StandardScaler()),
    ('MinMaxScaler', MinMaxScaler()),
    ('RobustScaler', RobustScaler()),
]

hs_results: list = []
for scaler_name, scaler_obj in scaler_configs:
    if scaler_obj is None:
        X_tr_s: np.ndarray = X_tr_hs
        X_te_s: np.ndarray = X_te_hs
    else:
        X_tr_s = scaler_obj.fit_transform(X_tr_hs)
        X_te_s = scaler_obj.transform(X_te_hs)

    ridge_hs = Ridge(alpha=RIDGE_ALPHA)
    ridge_hs.fit(X_tr_s, y_tr_hs)
    r2_te: float = ridge_hs.score(X_te_s, y_te_hs)
    mae_te: float = float(np.mean(np.abs(y_te_hs - ridge_hs.predict(X_te_s))))
    hs_results.append({'Scaler': scaler_name, 'Test R²': r2_te, 'Test MAE': mae_te})

hs_df: pd.DataFrame = pd.DataFrame(hs_results).set_index('Scaler')
print(hs_df.round(5).to_string())
print()
print('Ridge with L2 regularisation is scale-sensitive: unscaled features lead to')
print('poorly calibrated regularisation (features on different scales are penalised unequally).')

In [ ]:
# ── Encoder comparison on synthetic mixed dataset ─────────────────────────────
print('=== Encoder Comparison on Synthetic Mixed Dataset (Ridge, test R²) ===')

ridge_syn = Ridge(alpha=RIDGE_ALPHA)

encoder_configs: list = [
    ('OrdinalEncoder (wrong — treats zone as ordered)', OrdinalEncoder()),
    ('OneHotEncoder (correct for nominal)', OneHotEncoder(drop=True)),
    ('TargetEncoder (correct, smoothed)', TargetEncoder(smoothing=10.0)),
]

enc_results: list = []
for enc_name, enc_obj in encoder_configs:
    pipe_enc = TabularPipeline(
        num_cols=num_idx, cat_cols=cat_idx,
        scaler=StandardScaler(), encoder=enc_obj,
    )
    if isinstance(enc_obj, TargetEncoder):
        X_tr_enc: np.ndarray = pipe_enc.fit_transform(X_tr_syn_raw, y_tr_syn)
    else:
        X_tr_enc = pipe_enc.fit_transform(X_tr_syn_raw)
    X_te_enc: np.ndarray = pipe_enc.transform(X_te_syn_raw)

    ridge_syn.fit(X_tr_enc, y_tr_syn)
    r2: float = ridge_syn.score(X_te_enc, y_te_syn)
    n_out_cols: int = X_tr_enc.shape[1]
    enc_results.append({'Encoder': enc_name, 'Test R²': r2, 'Output cols': n_out_cols})

enc_df: pd.DataFrame = pd.DataFrame(enc_results).set_index('Encoder')
print(enc_df.round(5).to_string())
print()
print('OrdinalEncoder imposes a false ordering on nominal categories (zone: urban > suburban ...)')
print('which Ridge partially exploits. OneHotEncoder and TargetEncoder handle this correctly.')

In [ ]:
# ── Validate against sklearn Pipeline + ColumnTransformer ─────────────────────
print('=== Scratch Pipeline vs sklearn Pipeline + ColumnTransformer ===')

# Our pipeline on California Housing (StandardScaler)
pipe_ours = TabularPipeline(
    num_cols=list(range(X_tr_hs.shape[1])), cat_cols=[],
    scaler=StandardScaler(), encoder=None,
)
X_tr_ours: np.ndarray = pipe_ours.fit_transform(X_tr_hs)
X_te_ours: np.ndarray = pipe_ours.transform(X_te_hs)
ridge_val = Ridge(alpha=RIDGE_ALPHA)
ridge_val.fit(X_tr_ours, y_tr_hs)
r2_ours: float = ridge_val.score(X_te_ours, y_te_hs)

# sklearn Pipeline
sk_pipe = SklearnPipeline([
    ('scaler', SkStandardScaler()),
    ('ridge', Ridge(alpha=RIDGE_ALPHA)),
])
sk_pipe.fit(X_tr_hs, y_tr_hs)
r2_sk: float = sk_pipe.score(X_te_hs, y_te_hs)

print(f'Scratch Pipeline  test R²: {r2_ours:.6f}')
print(f'sklearn Pipeline  test R²: {r2_sk:.6f}')
print(f'Difference               : {abs(r2_ours - r2_sk):.2e}')
assert abs(r2_ours - r2_sk) < 1e-6, 'Pipelines diverge!'
print('\nPipelines match sklearn ✓')

---
## Part 4 — Analysis: Scaling Effects, Feature Selection & Leakage Quantification

In [ ]:
# ── Visualise effect of scalers on feature distributions ─────────────────────
feature_idx_viz: int = 4   # AveOccup — most skewed California Housing feature
feat_name_viz: str = feature_cols_hs[feature_idx_viz]
raw_vals: np.ndarray = X_tr_hs[:, feature_idx_viz]

ss_viz = StandardScaler().fit(X_tr_hs)
mm_viz = MinMaxScaler().fit(X_tr_hs)
rb_viz = RobustScaler().fit(X_tr_hs)

scaled_versions: list = [
    ('Raw', raw_vals),
    ('Standard', ss_viz.transform(X_tr_hs)[:, feature_idx_viz]),
    ('MinMax',   mm_viz.transform(X_tr_hs)[:, feature_idx_viz]),
    ('Robust',   rb_viz.transform(X_tr_hs)[:, feature_idx_viz]),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, vals) in zip(axes, scaled_versions):
    ax.hist(vals, bins=50, color='steelblue', edgecolor='none')
    p5 = float(np.percentile(vals, 5))
    p95 = float(np.percentile(vals, 95))
    ax.axvline(p5,  color='red', ls='--', lw=1.5, label=f'5th={p5:.1f}')
    ax.axvline(p95, color='orange', ls='--', lw=1.5, label=f'95th={p95:.1f}')
    ax.set_title(f'{name}\n{feat_name_viz}', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_ylabel('Count')

plt.suptitle(
    'Effect of Scaling on a Skewed Feature (AveOccup): '
    'RobustScaler compresses the long tail', fontsize=10)
plt.tight_layout()
plt.savefig('04_03_scaler_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Summary statistics table for each scaler ─────────────────────────────────
print()
print('Scaler comparison — summary statistics for AveOccup:')
header = f'{"Scaler":<15} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8} {"IQR":>8}'
print(header)
print('-' * 56)
for name, scaler_cls in [("Standard", StandardScaler), ("MinMax", MinMaxScaler), ("Robust", RobustScaler)]:
    sc = scaler_cls()
    sc.fit(X_train[:, feat_idx:feat_idx+1])
    col = sc.transform(X_test[:, feat_idx:feat_idx+1]).ravel()
    iqr = float(np.percentile(col, 75) - np.percentile(col, 25))
    print(f'{name:<15} {col.mean():>8.3f} {col.std():>8.3f} {col.min():>8.3f} {col.max():>8.3f} {iqr:>8.3f}')

In [ ]:
# ── Feature selection: MI vs L1 comparison ────────────────────────────────────
print('=== Feature Selection: Mutual Information vs L1 (Lasso) ===')

# Scale first
ss_fs = StandardScaler()
X_tr_fs: np.ndarray = ss_fs.fit_transform(X_tr_hs)
X_te_fs: np.ndarray = ss_fs.transform(X_te_hs)

# MI selection: top k features
mi_results: list = []
for k_sel in range(1, X_tr_hs.shape[1] + 1):
    X_tr_mi_k, mi_scores_k, sel_names = select_features_mi(
        X_tr_fs, y_tr_hs, k=k_sel, feature_names=feature_cols_hs)
    X_te_mi_k = X_te_fs[:, np.argsort(mi_scores_k)[::-1][:k_sel]]
    ridge_mi = Ridge(alpha=RIDGE_ALPHA)
    ridge_mi.fit(X_tr_mi_k, y_tr_hs)
    r2_mi: float = ridge_mi.score(X_te_mi_k, y_te_hs)
    mi_results.append((k_sel, r2_mi, sel_names))

# L1 selection via Lasso
lasso = Lasso(alpha=LASSO_ALPHA, max_iter=5000)
lasso.fit(X_tr_fs, y_tr_hs)
lasso_coefs: np.ndarray = np.abs(lasso.coef_)
l1_selected: list = [
    feature_cols_hs[i] for i in range(len(feature_cols_hs))
    if lasso_coefs[i] > 0
]
l1_mask: np.ndarray = lasso_coefs > 0
ridge_l1 = Ridge(alpha=RIDGE_ALPHA)
ridge_l1.fit(X_tr_fs[:, l1_mask], y_tr_hs)
r2_l1: float = ridge_l1.score(X_te_fs[:, l1_mask], y_te_hs)

print(f'L1 (Lasso alpha={LASSO_ALPHA}) selected features ({l1_mask.sum()}/{len(feature_cols_hs)}): '
      f'{l1_selected}')
print(f'L1 selected → Ridge test R²: {r2_l1:.5f}')
print()
print('MI selection results (k features):')
print(f'{"k":>4} {"Test R²":>10} {"Features selected"}')
print('-' * 60)
for k_val, r2_val, names in mi_results:
    print(f'{k_val:>4} {r2_val:>10.5f}  {names}')

# Plot MI R² vs k
k_vals = [r[0] for r in mi_results]
r2_vals = [r[1] for r in mi_results]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_vals, r2_vals, 'bo-', lw=2, ms=8, label='MI top-k features')
ax.axhline(r2_l1, color='red', ls='--', lw=2,
           label=f'L1 selected ({l1_mask.sum()} features, R²={r2_l1:.4f})')
ax.axhline(r2_vals[-1], color='grey', ls=':', lw=1.5,
           label=f'All features (R²={r2_vals[-1]:.4f})')
ax.set_xlabel('Number of MI-selected features')
ax.set_ylabel('Test R²')
ax.set_title('Feature Selection: MI vs L1 (California Housing → Ridge)')
ax.set_xticks(k_vals)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_03_feature_selection.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Target encoding leakage: cross-fold vs global encoding ────────────────────
print('=== Target Encoding: Leakage Risk and Cross-Fold Prevention ===')
print()
print('TargetEncoder fitted on the full training set leaks target info into the')
print('training features (the category mean = a perfect predictor of the target).')
print('In k-fold CV this leaks because the encoder sees the val fold targets.')
print('Solution: use a nested CV or encode each fold separately (cross-fold encoding).')
print()


kf_te = KFold(n_splits=CV_K, shuffle=True, random_state=SEED)
scores_leaky_te: list = []   # TargetEncoder fit on full train before CV split
scores_clean_te: list = []   # TargetEncoder fit only on inner fold

for tr_idx_te, vl_idx_te in kf_te.split(X_tr_syn_raw):
    X_cv_tr: np.ndarray = X_tr_syn_raw[tr_idx_te]
    X_cv_vl: np.ndarray = X_tr_syn_raw[vl_idx_te]
    y_cv_tr: np.ndarray = y_tr_syn[tr_idx_te]
    y_cv_vl: np.ndarray = y_tr_syn[vl_idx_te]

    # LEAKY: fit TargetEncoder on entire train set, then split
    te_leaky = TargetEncoder(smoothing=10.0)
    pipe_leaky = TabularPipeline(
        num_cols=num_idx, cat_cols=cat_idx,
        scaler=StandardScaler(), encoder=te_leaky,
    )
    pipe_leaky.fit(X_tr_syn_raw, y_tr_syn)  # fits on ALL training data!
    ridge_le = Ridge(alpha=RIDGE_ALPHA)
    ridge_le.fit(pipe_leaky.transform(X_cv_tr), y_cv_tr)
    scores_leaky_te.append(ridge_le.score(pipe_leaky.transform(X_cv_vl), y_cv_vl))

    # CLEAN: fit TargetEncoder only on inner CV training fold
    te_clean = TargetEncoder(smoothing=10.0)
    pipe_clean = TabularPipeline(
        num_cols=num_idx, cat_cols=cat_idx,
        scaler=StandardScaler(), encoder=te_clean,
    )
    pipe_clean.fit(X_cv_tr, y_cv_tr)  # fits only on inner train fold
    ridge_cl = Ridge(alpha=RIDGE_ALPHA)
    ridge_cl.fit(pipe_clean.transform(X_cv_tr), y_cv_tr)
    scores_clean_te.append(ridge_cl.score(pipe_clean.transform(X_cv_vl), y_cv_vl))

mean_leaky_te: float = float(np.mean(scores_leaky_te))
mean_clean_te: float = float(np.mean(scores_clean_te))
print(f'Leaky TargetEncoder CV R²: {mean_leaky_te:.5f} ± {np.std(scores_leaky_te):.5f}')
print(f'Clean TargetEncoder CV R²: {mean_clean_te:.5f} ± {np.std(scores_clean_te):.5f}')
print(f'Optimistic bias           : {mean_leaky_te - mean_clean_te:+.5f}')
print()
print('TargetEncoder leakage is far larger than StandardScaler leakage because the')
print('category means are computed from the target itself — they are directly correlated.')

### 1.4 Binning & Discretisation

Continuous features can be converted to **ordinal bin indices** — a useful step
when downstream models (logistic regression, Naive Bayes) cannot learn non-linear
boundaries on their own.  Two strategies:

- **Uniform** — equal-width intervals (sensitive to outliers)
- **Quantile** — equal-frequency intervals (robust; preserves class balance)


In [ ]:
# ── Binning & Discretisation from scratch ──────────────────────────────────────
class KBinsDiscretizer:
    '''Discretise continuous features into k equal-width or quantile-based bins.

    Adds a feature engineering primitive that converts numeric ranges to integer
    bin indices.  Useful as a preprocessing step before models that cannot learn
    non-linear boundaries on their own.

    Args:
        n_bins: Number of bins per feature.
        strategy: ``"uniform"`` (equal-width) or ``"quantile"`` (equal-frequency).
    '''

    def __init__(self, n_bins: int = 5, strategy: str = "uniform") -> None:
        '''Store hyperparameters; bin edges computed during fit.

        Args:
            n_bins: Number of bins per feature.
            strategy: Binning strategy — "uniform" or "quantile".
        '''
        self.n_bins = n_bins
        self.strategy = strategy
        self.bin_edges_: list = []

    def fit(self, X: np.ndarray) -> "KBinsDiscretizer":
        '''Compute bin edges for each feature column.

        Args:
            X: 2-D array of shape (n_samples, n_features).

        Returns:
            self
        '''
        self.bin_edges_ = []
        for j in range(X.shape[1]):
            col = X[:, j]
            if self.strategy == "uniform":
                edges = np.linspace(col.min(), col.max(), self.n_bins + 1)
            elif self.strategy == "quantile":
                quantiles = np.linspace(0, 100, self.n_bins + 1)
                edges = np.percentile(col, quantiles)
            else:
                raise ValueError(f"Unknown strategy: {self.strategy!r}")
            # Slightly widen outer edges so boundary points are always included
            edges[0] -= 1e-9
            edges[-1] += 1e-9
            self.bin_edges_.append(edges)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        '''Encode each feature as a zero-indexed bin integer.

        Args:
            X: 2-D array of shape (n_samples, n_features).

        Returns:
            Integer array of shape (n_samples, n_features).
        '''
        result = np.zeros_like(X, dtype=np.int64)
        for j, edges in enumerate(self.bin_edges_):
            result[:, j] = np.digitize(X[:, j], edges[1:])  # right-edge lookup
            result[:, j] = np.clip(result[:, j], 0, self.n_bins - 1)
        return result

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        '''Fit and transform in one pass.

        Args:
            X: 2-D array of shape (n_samples, n_features).

        Returns:
            Integer array of shape (n_samples, n_features).
        '''
        return self.fit(X).transform(X)


# ── Validate against sklearn KBinsDiscretizer ──────────────────────────────

rng_kbd = np.random.default_rng(SEED)
X_bins_demo = rng_kbd.standard_normal((400, 3))

for strategy in ["uniform", "quantile"]:
    kbd_scratch = KBinsDiscretizer(n_bins=6, strategy=strategy)
    Xt_scratch = kbd_scratch.fit_transform(X_bins_demo)

    sk_kbd = SkKBins(n_bins=6, encode="ordinal", strategy=strategy, subsample=None)
    Xt_sk = sk_kbd.fit_transform(X_bins_demo).astype(np.int64)

    match_pct = np.mean(Xt_scratch == Xt_sk) * 100
    print(f"[{strategy:8s}] Bin-index agreement with sklearn: {match_pct:.1f}%")

# Visualise one feature before and after binning
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(X_bins_demo[:, 0], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Original continuous feature")
kbd_q = KBinsDiscretizer(n_bins=6, strategy="quantile").fit(X_bins_demo)
axes[1].hist(kbd_q.transform(X_bins_demo)[:, 0], bins=6, color="coral",
             edgecolor="white", rwidth=0.8)
axes[1].set_title("After quantile binning (6 bins)")
plt.suptitle("KBinsDiscretizer: continuous → ordinal", fontweight="bold")
plt.tight_layout()
plt.show()
assert kbd_scratch.n_bins == 6, "n_bins should be stored correctly"
print("KBinsDiscretizer validation complete.")


### End-to-End CV: TabularPipeline + Ridge

A critical sanity check: does the pipeline actually prevent leakage and produce
valid held-out R²?  We sweep Ridge regularisation strengths under 5-fold CV
to confirm correctness and measure sensitivity to `alpha`.


In [ ]:
# ── End-to-end K-fold CV: TabularPipeline + Ridge on synthetic data ─────────


def pipeline_kfold_cv(
    X_num: np.ndarray,
    X_cat: np.ndarray,
    y: np.ndarray,
    num_col_indices: list,
    cat_col_indices: list,
    k: int = 5,
    alpha: float = 1.0,
) -> tuple:
    '''Run k-fold CV using TabularPipeline + Ridge; each fold prevents leakage.

    Fits the pipeline on training rows only, transforms validation rows, then
    scores with R².

    Args:
        X_num: Numeric feature matrix of shape (n_samples, n_num).
        X_cat: Categorical feature matrix of shape (n_samples, n_cat).
        y: Regression target vector of length n_samples.
        num_col_indices: Column positions corresponding to numeric features.
        cat_col_indices: Column positions corresponding to categorical features.
        k: Number of cross-validation folds.
        alpha: Ridge regularisation strength.

    Returns:
        Tuple of (mean_r2, std_r2) across folds.
    '''
    X_full = np.hstack([X_num, X_cat.astype(float)])
    n = len(y)
    fold_size = n // k
    idx = np.arange(n)
    rng_cv = np.random.default_rng(SEED)
    rng_cv.shuffle(idx)

    r2_scores: list = []
    for fold in range(k):
        val_idx = idx[fold * fold_size: (fold + 1) * fold_size]
        train_idx = np.concatenate([idx[:fold * fold_size],
                                    idx[(fold + 1) * fold_size:]])
        X_tr, X_val = X_full[train_idx], X_full[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        pipe = TabularPipeline(
            num_cols=num_col_indices,
            cat_cols=cat_col_indices,
            scaler=StandardScaler(),
            encoder=OneHotEncoder(drop=True),
        )
        X_tr_enc = pipe.fit_transform(X_tr, y_tr)
        X_val_enc = pipe.transform(X_val)

        model = Ridge(alpha=alpha)
        model.fit(X_tr_enc, y_tr)
        r2_scores.append(r2_score(y_val, model.predict(X_val_enc)))

    return float(np.mean(r2_scores)), float(np.std(r2_scores))


# Build combined column index lists from the synthetic dataset
X_num_arr = synth_df[num_features].values
X_cat_arr = synth_df[cat_features].values
y_arr = synth_df["price"].values
num_col_idx = list(range(X_num_arr.shape[1]))
cat_col_idx = list(range(X_num_arr.shape[1], X_num_arr.shape[1] + X_cat_arr.shape[1]))

print("TabularPipeline + Ridge  |  5-fold CV on synthetic dataset")
print(f"{'Alpha':>10s}  {'Mean R2':>8s}  {'Std R2':>8s}")
print("-" * 32)

cv_records: list = []
for alpha_val in [0.01, 0.1, 1.0, 10.0, 100.0]:
    mu, sigma = pipeline_kfold_cv(
        X_num_arr, X_cat_arr, y_arr,
        num_col_idx, cat_col_idx,
        k=5, alpha=alpha_val,
    )
    cv_records.append((alpha_val, mu, sigma))
    print(f"{alpha_val:>10.2f}  {mu:>8.4f}  {sigma:>8.4f}")

# Plot R2 vs alpha (log scale)
fig, ax = plt.subplots(figsize=(7, 4))
alphas_plot = [r[0] for r in cv_records]
means_plot  = [r[1] for r in cv_records]
stds_plot   = [r[2] for r in cv_records]
ax.errorbar(alphas_plot, means_plot, yerr=stds_plot,
            marker="o", capsize=5, color="steelblue", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Ridge alpha (log scale)", fontsize=11)
ax.set_ylabel("Mean R2 (5-fold CV)", fontsize=11)
ax.set_title("TabularPipeline + Ridge: CV R2 vs Regularisation",
             fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Pipeline CV benchmark complete.")


### Encoding Cardinality Analysis

As unique category count grows, OHE output width grows linearly while ordinal
encoding stays at width 1.  Target encoding maintains width 1 *and* preserves
signal — at the cost of requiring careful leakage avoidance.  This cell
quantifies the trade-off across cardinalities 2 → 100.


In [ ]:
# ── Encoding cardinality analysis: OHE vs Ordinal vs Target ─────────────────
# As category count grows, OHE output width grows linearly; ordinal stays at 1.
# Target encoding stays at width 1 AND preserves numeric signal — but requires
# careful cross-fitting to avoid leakage.


def encoding_cardinality_benchmark(
    cardinalities: list,
    n_samples: int = 800,
    noise: float = 0.1,
    seed: int = SEED,
) -> dict:
    '''Benchmark OHE vs Ordinal vs Target encoding across cardinality levels.

    For each cardinality ``c``, generates a single categorical feature with ``c``
    unique values, assigns a random numeric effect per category, adds Gaussian
    noise, and evaluates Ridge R² on a hold-out set for each encoder.

    Args:
        cardinalities: List of cardinality values to sweep.
        n_samples: Number of rows to generate per cardinality.
        noise: Standard deviation of Gaussian noise on the target.
        seed: Master random seed.

    Returns:
        Dict with keys ``"cardinality"``, ``"ohe_width"``, ``"ohe_r2"``,
        ``"ord_r2"``, ``"tgt_r2"``.
    '''

    rng_b = np.random.default_rng(seed)
    results: dict = {
        "cardinality": [], "ohe_width": [],
        "ohe_r2": [], "ord_r2": [], "tgt_r2": [],
    }

    for c in cardinalities:
        cats = np.array([str(i) for i in range(c)])
        idx = rng_b.integers(0, c, n_samples)
        X_cat_raw = cats[idx].reshape(-1, 1)            # (n, 1) string array
        effects = rng_b.standard_normal(c)
        y_c = effects[idx] + rng_b.normal(0, noise, n_samples)

        # 80 / 20 split
        split = int(0.8 * n_samples)
        perm = rng_b.permutation(n_samples)
        tr, val = perm[:split], perm[split:]
        X_tr, X_val = X_cat_raw[tr], X_cat_raw[val]
        y_tr, y_val = y_c[tr], y_c[val]

        # One-Hot Encoding
        ohe = OneHotEncoder(drop=False)
        Xtr_ohe = ohe.fit_transform(X_tr)
        Xval_ohe = ohe.transform(X_val)
        r_ohe = r2_score(y_val, Ridge(alpha=1.0).fit(Xtr_ohe, y_tr).predict(Xval_ohe))

        # Ordinal Encoding
        oe = OrdinalEncoder()
        Xtr_ord = oe.fit_transform(X_tr)
        Xval_ord = oe.transform(X_val)
        r_ord = r2_score(y_val, Ridge(alpha=1.0).fit(Xtr_ord, y_tr).predict(Xval_ord))

        # Target Encoding (fit on train only — no leakage)
        te = TargetEncoder(smoothing=5.0)
        Xtr_tgt = te.fit_transform(X_tr, y_tr)
        Xval_tgt = te.transform(X_val)
        r_tgt = r2_score(y_val, Ridge(alpha=1.0).fit(Xtr_tgt, y_tr).predict(Xval_tgt))

        results["cardinality"].append(c)
        results["ohe_width"].append(Xtr_ohe.shape[1])
        results["ohe_r2"].append(r_ohe)
        results["ord_r2"].append(r_ord)
        results["tgt_r2"].append(r_tgt)

    return results


card_bench = encoding_cardinality_benchmark([2, 5, 10, 20, 50, 100])

print(f"{'Card':>5s}  {'OHE width':>9s}  {'OHE R2':>8s}  {'Ord R2':>8s}  {'Tgt R2':>8s}")
print("-" * 48)
for i, c in enumerate(card_bench["cardinality"]):
    print(
        f"{c:5d}  {card_bench['ohe_width'][i]:9d}  "
        f"{card_bench['ohe_r2'][i]:8.4f}  "
        f"{card_bench['ord_r2'][i]:8.4f}  "
        f"{card_bench['tgt_r2'][i]:8.4f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cards = card_bench["cardinality"]

axes[0].plot(cards, card_bench["ohe_width"], marker="o", color="steelblue",
             linewidth=2, label="OHE output columns")
axes[0].set_xlabel("Cardinality", fontsize=11)
axes[0].set_ylabel("Output columns", fontsize=11)
axes[0].set_title("OHE Dimensionality Explosion", fontsize=12, fontweight="bold")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(cards, card_bench["ohe_r2"], marker="o", label="OHE",     color="steelblue", linewidth=2)
axes[1].plot(cards, card_bench["ord_r2"], marker="s", label="Ordinal", color="coral",     linewidth=2)
axes[1].plot(cards, card_bench["tgt_r2"], marker="^", label="Target",  color="green",     linewidth=2)
axes[1].set_xlabel("Cardinality", fontsize=11)
axes[1].set_ylabel("R2 (hold-out)", fontsize=11)
axes[1].set_title("Encoding Quality vs Cardinality", fontsize=12, fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Encoding cardinality analysis complete.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Always fit transformers on training data only.** Fitting a scaler or encoder on the entire dataset before splitting leaks test-set statistics into the model. sklearn's Pipeline enforces this automatically by chaining fit and transform steps; our `TabularPipeline` does the same. Target encoding leakage is particularly severe because the category means are directly derived from the target variable.

2. **Scaler choice matters for regularised models.** Ridge and Lasso are scale-sensitive: without scaling, features with large magnitudes dominate the regularisation penalty and the gradient. RobustScaler is the best default when features have heavy tails or outliers — it uses the median and IQR rather than the mean and std.

3. **Encoding strategy must respect the data semantics.** Ordinal encoding imposes an artificial ordering on nominal categories (e.g., `urban > suburban` makes no sense), which misleads models that exploit ordering. One-hot encoding is correct for unordered categories; target encoding is effective for high-cardinality categories when applied correctly inside a CV loop.

4. **Feature selection adds value beyond dimension reduction.** Removing low-MI features before fitting can improve test performance by reducing noise. MI is a fast, model-free filter; L1 regularisation is an embedded method that simultaneously selects and fits. Both techniques agree on the most informative features for California Housing.

5. **The sklearn Pipeline abstraction is a best practice.** By wrapping preprocessing and model into a single object, Pipeline enables cross-validation, grid search, and deployment without any risk of fitting the preprocessor on held-out data. Our `TabularPipeline` replicates this philosophy from scratch.

### What's Next

→ **04-04 (Data Augmentation & Color Spaces)** extends feature engineering to image data, where domain-informed transforms (flips, rotations, color jitter) serve as implicit regularisation.